In [1]:
import sys
from pathlib import Path

# Add the project root to Python path
project_root = Path.cwd().parent  # Go up one level from notebooks/
sys.path.insert(0, str(project_root))

# Now you can import from src/
from src import config
from src.feature_extraction import EmbeddingExtractor
from src.data_splitting import DataSplitter
from src.train import ModelTrainer

print(config.DATA_DIR)

2026-02-12 10:13:58.620129: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


ModuleNotFoundError: No module named 'data_splitting'

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pickle
import os
import time
import openl3   
from tqdm import tqdm

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score, precision_score, classification_report

from sklearn.model_selection import train_test_split
 

In [3]:
dataOLD = np.load('../features/embeddings/openl3_embeddings_6144_all.npz', allow_pickle=True)
dataNEW = np.load('../features/embeddings/openl3_6144_v2.npz', allow_pickle=True)

In [4]:
sort_idx_old = np.argsort(dataOLD['files'])

# Apply same ordering to all fields
X_old_sorted = dataOLD['X'][sort_idx_old]
y_old_sorted = dataOLD['y'][sort_idx_old]
files_old_sorted = dataOLD['files'][sort_idx_old]

sort_idx_new = np.argsort(dataNEW['files'])

# Apply same ordering to all fields
X_new_sorted = dataNEW['X'][sort_idx_new]
y_new_sorted = dataNEW['y_crackle'][sort_idx_new]
files_new_sorted = dataNEW['files'][sort_idx_new]

In [6]:
y_old_sorted.sum() - y_new_sorted.sum()

0

In [1]:
2.9/.6

4.833333333333333

In [ ]:
def load_embeddings_from_npz(dataset, npz_filepath='embeddings_512_all.npz'):
    """
    Load embeddings from npz file for a specific RespiratoryDataset
    
    Args:
        dataset: RespiratoryDataset object
        npz_filepath: Path to the saved npz file
    
    Returns:
        X: Embeddings array for samples in dataset
        y: Labels array for samples in dataset
    """
    # Load the npz file
    data = np.load(npz_filepath, allow_pickle=True)
    X_all = data['X']
    y_all = data['y']
    files_all = data['files']
    
    # Get the filenames from the dataset
    dataset_files = np.array(dataset.files)
    
    # Find indices in the full dataset that match this dataset's files
    indices = []
    for filename in dataset_files:
        idx = np.where(files_all == filename)[0]
        if len(idx) > 0:
            indices.append(idx[0])
        else:
            raise ValueError(f"File {filename} not found in embeddings file")
    
    indices = np.array(indices)
    
    # Extract X and y for these files in the same order as dataset.files
    X = X_all[indices]
    y = y_all[indices]
    
    return X, y

In [ ]:
# Running Random Forest model on pre-established 60/20/20 train/val/test split

# Determine output folder
streams_str = []

output_folder = f"Forest_Record_01"
output_path = os.path.join(config.dir_testTrainData, output_folder)
if not os.path.exists(output_path):
    os.makedirs(output_path)


# Pass model to datasets
train_dataset = RespiratoryDataset(split_name='train')
val_dataset = RespiratoryDataset(split_name='val')
test_dataset = RespiratoryDataset(split_name='test')

print("Loading OpenL3 embeddings...")
X_train, y_train = load_embeddings_from_npz(train_dataset,npz_filepath='embeddings_6144_all.npz')
X_val, y_val = load_embeddings_from_npz(val_dataset,npz_filepath='embeddings_6144_all.npz')
X_test, y_test = load_embeddings_from_npz(test_dataset,npz_filepath='embeddings_6144_all.npz')

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Sample embedding shape: {X_train[0].shape}\n")

#Train random forest
rf = RandomForestClassifier(
    n_estimators=50,
    class_weight={0:1,1:3},
    max_depth=8,
    min_samples_split=10,  # Prevent trees from overfitting to rare class
    min_samples_leaf=5,    # Ensure leaf nodes have enough samples
    random_state=42
)
rf.fit(X_train, y_train)

train_predictions = rf.predict(X_train)
val_predictions = rf.predict(X_val)
test_predictions = rf.predict(X_test)

# Training metrics
train_accuracy = rf.score(X_train, y_train)
train_f1 = f1_score(y_train, train_predictions)
train_recall = recall_score(y_train, train_predictions)
train_precision = precision_score(y_train, train_predictions)

# Validation metrics
val_accuracy = rf.score(X_val, y_val)
val_f1 = f1_score(y_val, val_predictions)
val_recall = recall_score(y_val, val_predictions)
val_precision = precision_score(y_val, val_predictions)

# Test metrics
test_accuracy = rf.score(X_test, y_test)
test_f1 = f1_score(y_test, test_predictions)
test_recall = recall_score(y_test, test_predictions)
test_precision = precision_score(y_test, test_predictions)

print(f'Train Accuracy: {train_accuracy:.4f}')
print(f'Train F1: {train_f1:.4f}')
print(f'Train Recall: {train_recall:.4f}')
print(f'Train Precision: {train_precision:.4f}')
print(f'Train Split (Actual): {sum(y_train)/len(y_train):.2f}, Split (Predicted): {sum(train_predictions)/len(train_predictions):.2f}\n')

print(f'Val Accuracy: {val_accuracy:.4f}')
print(f'Val F1: {val_f1:.4f}')
print(f'Val Recall: {val_recall:.4f}')
print(f'Val Precision: {val_precision:.4f}\n')
print(f'Val Split (Actual): {sum(y_val)/len(y_val):.2f}, Split (Predicted): {sum(val_predictions)/len(val_predictions):.2f}\n')


print(f'Test Accuracy: {test_accuracy:.4f}')
print(f'Test F1: {test_f1:.4f}')
print(f'Test Recall: {test_recall:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test Split (Actual): {sum(y_test)/len(y_test):.2f}, Split (Predicted): {sum(test_predictions)/len(test_predictions):.2f}\n')

# Or use classification_report for a nice summary
print(classification_report(y_test, test_predictions))

importances = rf.feature_importances_

with open('model_results.txt', 'w') as f:
    f.write(f'Test Accuracy: {test_accuracy:.4f}\n')
    f.write(f'Test F1: {test_f1:.4f}\n')
    f.write(f'Test Recall: {test_recall:.4f}\n')
    f.write(f'Test Precision: {test_precision:.4f}\n')
    f.write('\n' + classification_report(y_test, test_predictions) + '\n')
    
    for i, imp in enumerate(importances):
        f.write(f'Feature {i}: {imp:.4f}\n')

print("Results saved to model_results.txt")

Loading OpenL3 embeddings...
X_train shape: (1023, 6144)
X_val shape: (214, 6144)
X_test shape: (483, 6144)
Sample embedding shape: (6144,)

Train Accuracy: 0.9150
Train F1: 0.9049
Train Recall: 1.0000
Train Precision: 0.8263
Train Split (Actual): 0.40, Split (Predicted): 0.49

Val Accuracy: 0.7009
Val F1: 0.6768
Val Recall: 0.7882
Val Precision: 0.5929

Val Split (Actual): 0.40, Split (Predicted): 0.53

Test Accuracy: 0.6646
Test F1: 0.5291
Test Recall: 0.5482
Test Precision: 0.5112
Test Split (Actual): 0.34, Split (Predicted): 0.37

              precision    recall  f1-score   support

           0       0.75      0.73      0.74       317
           1       0.51      0.55      0.53       166

    accuracy                           0.66       483
   macro avg       0.63      0.64      0.63       483
weighted avg       0.67      0.66      0.67       483

Results saved to model_results.txt


In [ ]:
# Running grid search to find best parameters for training set (using 5-fold validation for each parameter grouping)

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, make_scorer

# Define parameter grid to search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [8, 10, 12, 15, None],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}

# Create base model
rf = RandomForestClassifier(class_weight={0:1, 1:3}, random_state=42)

# Create GridSearchCV
grid_search = GridSearchCV(
    rf, 
    param_grid, 
    cv=5,  # 5-fold cross-validation
    scoring=make_scorer(f1_score),  # Score by F1, not accuracy
    n_jobs=-1,  # Use all CPU cores
    verbose=1
)

# Fit on training data
grid_search.fit(X_train, y_train)

# Best parameters and score
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV F1 score: {grid_search.best_score_:.4f}")

# Evaluate best model on val and test
best_rf = grid_search.best_estimator_
val_f1 = f1_score(y_val, best_rf.predict(X_val))
test_f1 = f1_score(y_test, best_rf.predict(X_test))

print(f"Val F1: {val_f1:.4f}")
print(f"Test F1: {test_f1:.4f}")

Fitting 5 folds for each of 270 candidates, totalling 1350 fits
Best parameters: {'max_depth': 8, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 5, 'n_estimators': 200}
Best CV F1 score: 0.6382
Val F1: 0.6465
Test F1: 0.4969


In [23]:
# Splitting data into 70/30 train/test groups with stratification based on patient label

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, make_scorer


def get_patient_ids_for_all_samples(all_files):
    """Extract patient ID (first 3 characters) from filenames"""
    patient_ids = np.array([filename[:3] for filename in all_files])
    return patient_ids

# Load all embeddings
data = np.load('embeddings_6144_all.npz', allow_pickle=True)
X_all = data['X']
y_all = data['y']
all_files = data['files']

# Get patient IDs from filenames
patient_ids_all = get_patient_ids_for_all_samples(all_files)

# Get unique patients and their labels
unique_patients = np.unique(patient_ids_all)
patient_labels = []
for patient in unique_patients:
    patient_mask = patient_ids_all == patient
    # Use majority class label for this patient
    patient_label = np.bincount(y_all[patient_mask]).argmax()
    patient_labels.append(patient_label)

patient_labels = np.array(patient_labels)

# Split patients with stratification
train_patients, test_patients = train_test_split(
    unique_patients,
    test_size=0.3,
    random_state=42,
    stratify=patient_labels
)

# Get sample indices for each split
train_mask = np.isin(patient_ids_all, train_patients)
test_mask = np.isin(patient_ids_all, test_patients)

X_train = X_all[train_mask]
y_train = y_all[train_mask]
X_test = X_all[test_mask]
y_test = y_all[test_mask]

# Check distributions
print(f"Train minority %: {(y_train == 1).sum() / len(y_train):.2%}")
print(f"Test minority %: {(y_test == 1).sum() / len(y_test):.2%}")
print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Train minority %: 39.23%
Test minority %: 37.43%
Train samples: 1175, Test samples: 545


In [ ]:
from sklearn.model_selection import GroupKFold

patient_ids_train = get_patient_ids_for_all_samples(all_files[train_mask])

class_weight = {0: 1, 1: 3}

gkf = GroupKFold(n_splits=5)
rf = RandomForestClassifier(class_weight=class_weight, random_state=42)

# Define parameter grid to search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [8, 10, 12, 15, None],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}

grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=gkf,
    scoring=make_scorer(f1_score),
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train, groups=patient_ids_train)

# Best parameters and score
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV F1 score: {grid_search.best_score_:.4f}")

# Evaluate best model on val and test
best_rf = grid_search.best_estimator_
test_f1 = f1_score(y_test, best_rf.predict(X_test_scaled))

print(f"Test F1: {test_f1:.4f}")

Fitting 5 folds for each of 270 candidates, totalling 1350 fits
Best parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'min_samples_split': 20, 'n_estimators': 200}
Best CV F1 score: 0.6238
Test F1: 0.7106


In [24]:
#Trying Logistic Regression

from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(class_weight=class_weight, max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
test_f1 = f1_score(y_test, lr.predict(X_test_scaled))

print(f"LR Test F1: {test_f1:.4f}")

LR Test F1: 0.6811


In [25]:
# Trying SVM

from sklearn.svm import SVC

svm = SVC(class_weight='balanced', kernel='rbf', random_state=42)
svm.fit(X_train_scaled, y_train)
test_f1 = f1_score(y_test, svm.predict(X_test_scaled))

print(f"SVM Test F1: {test_f1:.4f}")

SVM Test F1: 0.7213


In [ ]:
#Trying MLP

from sklearn.neural_network import MLPClassifier

sample_weights = np.array([class_weight[label] for label in y_train])
mlp = MLPClassifier(hidden_layer_sizes=(128, 64), random_state=42)
mlp.fit(X_train, y_train, sample_weight=sample_weights)
test_f1 = f1_score(y_test, mlp.predict(X_test))

print(f"MLP Test F1: {test_f1:.4f}")

MLP Test F1: 0.6773


In [19]:
import scipy.signal as signal

def rawFilter(raw_audio, sample_rate, filter_order, filter_lowcut=80, filter_highcut=1800, btype="bandpass"):

    
    if btype == "bandpass":
        sos = signal.butter(filter_order, [filter_lowcut, filter_highcut], btype="bandpass",fs = sample_rate, output = 'sos')

    elif btype == "highpass":
        sos = signal.butter(filter_order, filter_lowcut, btype="highpass", fs=sample_rate,output='sos')

    elif btype == "lowpass":
        sos = signal.butter(filter_order, filter_highcut, btype="lowpass", fs=sample_rate,output='sos')

    audio = signal.sosfiltfilt(sos, raw_audio)

    return audio

In [ ]:
import librosa

dataFolder = "../Samples"

data_files = sorted(os.listdir(dataFolder))
#data_files = ['Eko_p005_t02_healthy_20250228.wav']
#data_files = ['Eko_p002_t01_severe_20250221.wav']
#data_files = ['Eko_p003_t01_moderate_20250227.wav']

for data_file in data_files:

    #if not a wav file, skip
    if ".wav" not in data_file:
        continue

    print(data_file)

    # path
    wavFilepath = f"{dataFolder}/{data_file}"

    # load data
    raw_audio, sample_rate = librosa.load(path=wavFilepath, sr=4000)

    #full_window = 2.5
    clipLength = 1.0
    hop = 0.5

    audio_data = []

    startWindow = 0
    endWindow = clipLength * sample_rate

    while startWindow + (clipLength * sample_rate) < len(raw_audio):
        clip = raw_audio[startWindow:endWindow]
        clip = rawFilter(clip, sample_rate, config.filter_order,
                                     config.filter_lowcut, config.filter_highcut, btype=config.filter_btype)
        
        audio_data.append(clip)

        startWindow += int(hop*sample_rate)
        endWindow = int(startWindow + clipLength * sample_rate)

    samples = []
    embeddings = []

    for segment in audio_data:
        
        embedding, ts = openl3.get_audio_embedding(
                segment, 4000, 
                content_type="env",
                embedding_size=6144,
                verbose=False
            )
        
        X = np.mean(embedding, axis=0)
        embeddings.append(X)

    # Concatenate all into (n_segments, 512)
    X_all = np.vstack(embeddings)

    X_all_scaled = scaler.transform(X_all_scaled)

    # Make predictions
    predictions_rf = best_rf.predict(X_all_scaled)
    print(f"RF Predictions: {predictions_rf}")
    print(np.mean(predictions_rf))
    
    predictions_svm = svm.predict(X_all_scaled)
    print(f"SVM Predictions: {predictions_svm}")
    print(np.mean(predictions_svm))

    predictions_lr = lr.predict(X_all_scaled)
    print(f"LR Predictions: {predictions_lr}")
    print(np.mean(predictions_lr))

    predictions_mlp = mlp.predict(X_all_scaled)
    print(f"MLP Predictions: {predictions_mlp}")
    print(np.mean(predictions_mlp))
    


    

Eko_p001_t01_mild_20250214.wav
RF Predictions: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0.0
SVM Predictions: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0.0
LR Predictions: [0 1 1 1 1 0 0 1 0 0 1 1 1 0 1 0 0 0 0 0 1 1 1 1 1 0 1 0]
0.5357142857142857
MLP Predictions: [0 1 1 0 1 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 1 1 0 0 0 0 0]
0.2857142857142857
Eko_p002_t01_severe_20250221.wav
RF Predictions: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0.0
SVM Predictions: [0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 1 1 1]
0.2857142857142857
LR Predictions: [1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1]
0.9285714285714286
MLP Predictions: [0 0 0 1 1 1 0 0 0 0 0 0 0 0 1 1 1 1 0 0 0 0 0 0 0 1 1 1]
0.35714285714285715
Eko_p002_t02_severe_20250221.wav
RF Predictions: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0.0
SVM Predictions: [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
0.07142857142857142
LR Predictions: [1 1 1 1 0

KeyboardInterrupt: 

In [31]:
decision_scores = svm.decision_function(X_test)
print(f"Min score: {decision_scores.min()}")
print(f"Max score: {decision_scores.max()}")
print(f"Mean score: {decision_scores.mean()}")

Min score: -2.1112703266481674
Max score: 1.7455956282384706
Mean score: -0.19993730661225345


In [30]:
train_pred = svm.predict(X_train)
print(f"Train predictions: {np.bincount(train_pred)}")
print(f"Actual train labels: {np.bincount(y_train)}")

Train predictions: [604 571]
Actual train labels: [714 461]


In [31]:
embedding.shape

(6, 6144)

In [19]:
from src.data_splitting import DataSplitter

def compare_class_weights_for_wheezes():
    """Compare different class weight strategies for wheeze detection"""
    
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report, f1_score, confusion_matrix
    
    # Load your data
    splitter = DataSplitter()

    split_data = splitter.load_split(embedding_size = 512, version='v1')
    X_train, y_train = split_data['X_train'], split_data['y_wheeze_train']
    X_test, y_test = split_data['X_test'], split_data['y_wheeze_test']

    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    
    weight_options = {
        'balanced': 'balanced',
        'moderate_4': {0: 1, 1: 4},
        'moderate_5': {0: 1, 1: 5},
        'aggressive_6': {0: 1, 1: 6},
        'aggressive_8': {0: 1, 1: 8},
        'very_aggressive_10': {0: 1, 1: 10},
    }
    
    results = {}
    for name, weights in weight_options.items():
        lr = LogisticRegression(class_weight=weights, max_iter=1000)
        lr.fit(X_train, y_train)
        y_pred = lr.predict(X_test)
        
        lr_scaled = LogisticRegression(class_weight=weights, max_iter=1000)
        lr_scaled.fit(X_train_scaled, y_train)
        y_pred_scaled = lr_scaled.predict(X_test_scaled)
        
        print(f"\n{'='*50}")
        print(f"Strategy: {name} - Weights: {weights}")
        print('='*50)
        print(classification_report(y_test, y_pred, target_names=['No Wheeze', 'Wheeze']))
        print(classification_report(y_test, y_pred_scaled, target_names=['No Wheeze', 'Wheeze']))
        print(confusion_matrix(y_test, y_pred))

        
        results[name] = {
            'f1': f1_score(y_test, y_pred),
            'recall': recall_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred),
            'f1_scaled': f1_score(y_test, y_pred_scaled),
            'recall_scaled': recall_score(y_test, y_pred_scaled),
            'precision_scaled': precision_score(y_test, y_pred_scaled)
        }
    
    # Summary
    print(f"\n{'='*50}")
    print("SUMMARY")
    print('='*50)
    for name, metrics in results.items():
        print(f"{name:20s} - F1: {metrics['f1']:.3f}, "
              f"Recall: {metrics['recall']:.3f}, "
              f"Precision: {metrics['precision']:.3f}, "
              f"F1 - scaled: {metrics['f1_scaled']:.3f}, "
              f"Recall - scaled: {metrics['recall_scaled']:.3f}, "
              f"Precision - scaled: {metrics['precision_scaled']:.3f} "
        )

In [20]:
compare_class_weights_for_wheezes()


Strategy: balanced - Weights: balanced
              precision    recall  f1-score   support

   No Wheeze       0.90      0.87      0.88       448
      Wheeze       0.46      0.54      0.50        97

    accuracy                           0.81       545
   macro avg       0.68      0.70      0.69       545
weighted avg       0.82      0.81      0.81       545

              precision    recall  f1-score   support

   No Wheeze       0.86      0.90      0.88       448
      Wheeze       0.43      0.34      0.38        97

    accuracy                           0.80       545
   macro avg       0.65      0.62      0.63       545
weighted avg       0.79      0.80      0.79       545

[[388  60]
 [ 45  52]]

Strategy: moderate_4 - Weights: {0: 1, 1: 4}
              precision    recall  f1-score   support

   No Wheeze       0.89      0.89      0.89       448
      Wheeze       0.49      0.47      0.48        97

    accuracy                           0.82       545
   macro avg       